# ResNet-18 (ImageNet-initialised)
Owner: Anders

ResNet-18 fine-tuned on the same frozen split as the small CNN, for comparison.

**Note:** `src/preprocessing.py` (Shashank's) is still a stub, so this notebook has a local, temporary
`load_and_pad` placeholder below, same as `03_cnn.ipynb`. Once `src/preprocessing.py` is implemented,
delete the placeholder cell and switch to `from src.preprocessing import load_and_pad`.

In [1]:
import sys
sys.path.append("..")

import kagglehub
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

SEED = 42
torch.manual_seed(SEED)

# Prefer a GPU: CUDA on other platforms, MPS (Metal) on Apple Silicon, else fall back to CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

Using device: mps


## Temporary preprocessing load and pad method

In [2]:
IMG_SIZE = 128

def load_and_pad(path: str, size: int = IMG_SIZE) -> np.ndarray:
    """TEMPORARY placeholder - mirrors what src/preprocessing.load_and_pad should do.
    Loads an RGB image, preserves aspect ratio, pads to a (size, size) square.
    """
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = size / max(w, h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    img = img.resize((new_w, new_h), Image.BILINEAR)

    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    paste_x = (size - new_w) // 2
    paste_y = (size - new_h) // 2
    canvas.paste(img, (paste_x, paste_y))

    arr = np.asarray(canvas, dtype=np.float32) / 255.0  # scale to [0, 1]
    return arr

## Load the frozen split and build a Dataset/DataLoader
Same `CellDataset` pattern as `03_cnn.ipynb`, but with one addition: since we're using
ImageNet-pretrained weights, inputs must be normalized with the same per-channel mean/std
ImageNet was trained with - otherwise the pretrained filters see pixel statistics they were
never trained on, and the pretrained weights stop being a useful starting point.

In [3]:
manifest = pd.read_csv("../data/splits/patient_split.csv")
print(manifest["split"].value_counts())

# The manifest only stores filename + category, not an absolute path (those differ per machine).
# Rebuild the path the same way 00_eda.ipynb resolved it: <kagglehub download root>/<category>/<filename>
download_path = Path(kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria"))
data_dir = download_path / "cell_images"
if (data_dir / "cell_images").exists():
    data_dir = data_dir / "cell_images"

CLASS_TO_IDX = {"Uninfected": 0, "Parasitized": 1}

# Per-channel mean/std ImageNet was trained with (required for pretrained ResNet-18 weights).
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class CellDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        path = self.data_dir / row["category"] / row["filename"]

        # Load the padded image as a (height, width, channels) array in [0, 1], apply the
        # ImageNet normalization, then rearrange to (channels, height, width) for PyTorch.
        img = load_and_pad(str(path))
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        img = torch.from_numpy(img.astype(np.float32)).permute(2, 0, 1)

        label = CLASS_TO_IDX[row["category"]]
        return img, label


train_ds = CellDataset(manifest[manifest["split"] == "train"], data_dir)
val_ds = CellDataset(manifest[manifest["split"] == "val"], data_dir)
test_ds = CellDataset(manifest[manifest["split"] == "test"], data_dir)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check: one batch
images, labels = next(iter(train_loader))
print(images.shape, images.dtype, labels.shape)

split
train    19124
val       4324
test      4110
Name: count, dtype: int64
torch.Size([64, 3, 128, 128]) torch.float32 torch.Size([64])


## ResNet-18 implementation
Load ImageNet-pretrained weights, then replace the final classification layer (`fc`), which was
built for ImageNet's 1000 classes, with one sized for our 2 classes. Everything else keeps its
pretrained weights as the starting point, and all layers are fine-tuned (not frozen).

In [4]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# fc is the final linear layer; in_features is however many features ResNet-18's conv backbone
# produces (512), so we keep that and only change out_features to our 2 classes.
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# Lower learning rate than the small CNN (1e-3): most of this network is already pretrained,
# so we want smaller updates that don't wreck the pretrained features early in training.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## Training loop (validation only - freeze the protocol before touching test_loader)

In [5]:
def run_epoch(loader, train: bool):
    model.train(mode=train)
    total_loss, correct, n = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        n += images.size(0)

    return total_loss / n, correct / n


NUM_EPOCHS = 3
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

Epoch 1: train_loss=0.1316 train_acc=0.9510 val_loss=0.1249 val_acc=0.9556
